# T3.3 EIT-002 Environmental Footprint Simulation

**READ.ME**

This notebook calls the databases and project defined in EIT-003. The notebook is mainly a copy of EIT-001 as no example datasets for Simulation Configurations were shared yet.

- Setting the project based on EIT-003
- Read in Process Simulation Hub data
- Integration in foreground database as scenario 
- LCA results disaggregated in:
    - Total impact
    - Impact per tool / machining operation

### 1. Setting the project

In [1]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
from bw2io.strategies.generic import set_code_by_activity_hash

12:08:02+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
config = Config()

In [3]:
bd.projects.set_current(config.PROJECT_NAME)

In [4]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

### 3. Import Simulation Data

This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [5]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [6]:
simulation_data_path = config.REED_PATH/'Data'/"SimulationData"/"WWR"/"T2472"/"data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [7]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    simulation_data_path,
    tool_descriptions_path,debug=True
)
print(sensordata)


Total CSV files imported: 4
                                          energy_consumption  Power_Active  \
company event_id             Tool_Number                                     
wwr     OF10004_G_TWB_7SZ421 0.0               199943.657180   6813.293128   
                             81.0               79448.274472   8209.350979   
                             82.0               23343.902749   8310.724871   
                             100.0              67116.527595   8319.371254   
                             102.0             118242.215355   9726.310414   
...                                                      ...           ...   
                             2694.0              4718.325664  13096.354965   
                             2695.0               959.320473   9700.993548   
                             2696.0              1756.356816  13036.875333   
                             2722.0               351.314922  10199.465472   
                             3350.0

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [8]:
print(ambiguous_company_tools)

   company  Tool_Number               op_type_candidates
0   goimek           31               [CHAMFER, MILLING]
1  komatsu            3         [MILLING, ROUGH MILLING]
2  komatsu           48         [MILLING, ROUGH MILLING]
3  komatsu           50    [ROUGH BORING, ROUGH MILLING]
4  komatsu           72  [FINISH MILLING, ROUGH MILLING]


In [9]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)

In [10]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)

### 4. Creating a scenario for Simulated Data integration

this section is missing a part about how to handle the processes that are not in hours?

In [11]:
from reed.swapping_data import (
    find_activities_by_name,
    build_scenarios_from_power_active,
    build_consumer_input_scenarios_from_hours,
    describe_scenarios,
    lcia_for_activity_with_scenario,
    find_subactivity_by_name_fragment,
    build_consumer_input_scenarios_from_hours_for_activity,
    get_product_id_for_activity,
)

Select which event you want to analyse

In [12]:
company = "wwr"
event_id = "OF10004_G_TWB_7SZ421"
product = "WWR total"
demand = 1

Start by selected the company and event_id where you want to swap in the sensordata

In [13]:
#Split sensordata by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[(company, event_id)]
)
print(tool_energy_by_company_and_event)

                                                          energy_consumption  \
company event_id             reed tool                                         
wwr     OF10004_G_TWB_7SZ421 boring, soraluce                   39487.347249   
                             drilling, soraluce                133918.229448   
                             generic machining, soraluce       472503.526821   
                             milling, soraluce                 550725.038966   

                                                              hours  \
company event_id             reed tool                                
wwr     OF10004_G_TWB_7SZ421 boring, soraluce              3.190278   
                             drilling, soraluce           14.090278   
                             generic machining, soraluce  55.911944   
                             milling, soraluce            51.766944   

                                                          Power_Active  
company event_id   


Find all processes present in the sensordata in the reed-database

In [14]:
tools = tool_energy_by_company_and_event.index.get_level_values("reed tool").unique()
found = {t: find_activities_by_name(reed, t, exact=False) for t in tools}
found

{'boring, soraluce': ['boring, soraluce' (hour, DE, None),
  'boring, soraluce' (hour, GB, None),
  'boring, soraluce' (hour, ES, None)],
 'drilling, soraluce': ['drilling, soraluce' (kilogram, GB, None),
  'drilling, soraluce' (hour, DE, None),
  'drilling, soraluce' (kilogram, DE, None),
  'drilling, soraluce' (hour, ES, None),
  'drilling, soraluce' (kilogram, ES, None),
  'drilling, soraluce' (hour, GB, None)],
 'generic machining, soraluce': ['generic machining, soraluce' (kilogram, DE, None),
  'generic machining, soraluce' (hour, GB, None),
  'generic machining, soraluce' (kilogram, GB, None),
  'generic machining, soraluce' (kilogram, ES, None),
  'generic machining, soraluce' (hour, ES, None),
  'generic machining, soraluce' (hour, DE, None)],
 'milling, soraluce': ['milling, soraluce' (kilogram, DE, None),
  'milling, soraluce' (hour, DE, None),
  'milling, soraluce' (hour, ES, None),
  'milling, soraluce' (hour, GB, None),
  'milling, soraluce' (kilogram, GB, None),
  'milli

Find subactivities called A3 for the product under study

In [15]:
product_activity = reed.get(name=product)

# Find the A3 activity inside KOMATSU total
a3_matches = find_subactivity_by_name_fragment(
    parent_activity=product_activity,
    name_fragment="A3",
)

print(a3_matches)
a3_activity = a3_matches[0]

['WWR A3: machining' (unit, DE, None)]


In [16]:
soraluce_matches = find_subactivity_by_name_fragment(
    parent_activity=a3_activity,
    name_fragment="soraluce machining",
)

print(soraluce_matches)

soraluce_activity = soraluce_matches[0]

['Soraluce machining, WWR' (unit, DE, None)]


In [17]:
#Identifying the electricity activity in bonsai to be able to compute the original electricity input amount for the activities found above
electricity = bonsai.get(name='production of electricity (market for)', location='DE', unit='TJ')

Create scenario where alle electricity inputs from the sensordata to the processes identified in found.

In [18]:
scenarios = build_scenarios_from_power_active(tool_energy_by_company_and_event, found, electricity)
scenarios


original  wwr_OF10004_G_TWB_7SZ421
product            activity                                              
318672587029032968 323362257478430720  0.000173                  0.044559
                   323362257461653504  0.000053                  0.034215
                   323362257369378816  0.000000                  0.030423
                   323362257436487680  0.000015                  0.038299

The input of products in found to consuming activities are found and used to build scenarios.

In [19]:
consumer_scenarios = build_consumer_input_scenarios_from_hours_for_activity(
    tool_energy_by_company_and_event,
    found,
    consumer_activity=soraluce_activity,
)
consumer_scenarios

,,original,wwr_OF10004_G_TWB_7SZ421
product,activity,,
323362257478430720,323362257612648448,3.190278,3.190278
323362257461653504,323362257612648448,4.410833,14.090278
323362257386156032,323362257612648448,65.591389,55.911944
323362257436487680,323362257612648448,51.766944,51.766944


In [20]:
full_scenario = pd.concat([scenarios, consumer_scenarios])

In [21]:
full_scenario_readable = describe_scenarios(full_scenario)
full_scenario_readable

,product_name,product_unit,activity_name,activity_unit,original,wwr_OF10004_G_TWB_7SZ421
0,production of electricity (market for),TJ,"boring, soraluce",hour,0.000173,0.044559
1,production of electricity (market for),TJ,"drilling, soraluce",hour,0.000053,0.034215
2,production of electricity (market for),TJ,"generic machining, soraluce",hour,0.000000,0.030423
3,production of electricity (market for),TJ,"milling, soraluce",hour,0.000015,0.038299
4,"boring, soraluce",hour,"Soraluce machining, WWR",unit,3.190278,3.190278
5,"drilling, soraluce",hour,"Soraluce machining, WWR",unit,4.410833,14.090278
6,"generic machining, soraluce",hour,"Soraluce machining, WWR",unit,65.591389,55.911944
7,"milling, soraluce",hour,"Soraluce machining, WWR",unit,51.766944,51.766944


### 5. Footprint Simulation Output

In [26]:
import importlib
import reed.LCA_functions
import json

importlib.reload(reed.LCA_functions)
from reed.LCA_functions import (get_machining_activity, simulation_footprint_output)

In [27]:
simulation_footprint_output_test = simulation_footprint_output(product_activity, lcia_method=config.lcia_gwp100, company=company, event_id=event_id)
print(json.dumps(simulation_footprint_output_test, indent=2))

Exported to simulation_wwr_OF10004_G_TWB_7SZ421_2026-06-17.json
{
  "product": "WWR total",
  "total_climate_impact_kg_co2eq": 40993.5972,
  "a1_raw_materials": {
    "total_kg_co2eq": 38133.3591
  },
  "a2_transport": {
    "total_kg_co2eq": 518.9271
  },
  "a3_machining": {
    "total_kg_co2eq": 1914.5552,
    "operations": [
      {
        "operation": "generic machining, soraluce",
        "score": 967.227137
      },
      {
        "operation": "boring, soraluce",
        "score": 85.672719
      },
      {
        "operation": "drilling, soraluce",
        "score": 75.782369
      },
      {
        "operation": "milling, soraluce",
        "score": 763.36839
      },
      {
        "operation": "soraluce machine, WWR",
        "score": 22.504584
      }
    ]
  }
}
